In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Data Cleansing
We will be using Spark to perform exploratory analysis and cleaning Data from a public Airbnb Dataset.

In [ ]:
output_path = '/home/jovyan/work/datasets/output/airbnb/clean_data'


# Input Data
Let's load the Airbnb dataset in.  
You can download data from any city from <a href="http://insideairbnb.com/get-the-data.html" target="_blank">Airbnb website.</a> The dataset we are going to use is the Detailed Listings data called listings.csv.gz
The one provided is the one from New York City, i recommend to store it in a folder such as:
```/home/jovyan/work/datasets/airbnb/listingsNY.csv```
Check the course documentation to know how to upload it.

# Load Data
First you need to load Data to a Dataframe, there are several options to consider: 
* header: true if the data contains a header , otherwise the header will be considered Data (Check input file to know which value to use)
* inferSchema: true to let spark infer the schema instead of provider it's types mannually. Set it to true
* multiLine: true to allow multiline texts in records. Set it to true
* escape: Ignore special characters in data, set it to '\"'
* quote: Character that encloses each value, set it to '\"'
## Expected output
* 38199 rows for the provided Dataset (may differ if you downloaded another version of another city)
* Typed dataset (Ensure not all of the fields are strings)
* 75 columns

In [2]:
from pyspark.sql.functions import split, col


file_path = f"/home/jovyan/work/datasets/airbnb/listingsNY.csv"

raw_df = spark.read.csv(file_path,
                         header="true", 
                          inferSchema="true", 
                          multiLine="true", 
                          escape='\"', 
                          quote='\"')

raw_df.count()

38199

In [3]:
raw_df.columns

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

# Cleaning columns
Keep only the following columns from this dataset.  Solo las que consideramos features y las que queremos predecir (precio)
```
columns_to_keep = [
    "host_is_superhost",
    "instant_bookable",
    "host_total_listings_count",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "price"
]
```
## Expected output
* 22 columns

In [11]:
columns_to_keep = [
    "host_is_superhost",
    "instant_bookable",
    "host_total_listings_count",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "price"
]

base_df = raw_df.select(columns_to_keep)



# Fixing Data Types
If you check the schema from the previous dataframe, you will be able to see that the **`price`** field was understand as a price. This is a pretty commons error when dealing with monetary amounts due to currency sumbols. We need it to be a number. So:
* Remove dollar sign with `translate` method  which replace the second parameter for the third one in the column indicated in the first parameter. In this case we should do `translate(col("price"), "$,", "")`
* cast column to double
* Use withColumn method for all that as follows: 
  *  `withColumn("price", translate(col("price), "$,", "").cast("double"))`
  
Once fixed the data type, we can build an histogram over the dataframe to see that there is a huge skew over the price column. Skew is one of the main issues in Spark.
Skew is when data is not balanced and it doesnt follows a normal distribution

In [23]:
from pyspark.sql.functions import col, translate

fixed_price_df = base_df.withColumn("price", translate(col("price"), "$", "").cast("double"))



# Data statistics
Two options:
* **`describe`**: Providers count, mean, stddev, min, max
* **`summary`**: Providers the ones from describe and the interquartile range (IQR) (25-50-75%)

In [ ]:
display(fixed_price_df.describe().toPandas())

DataFrame[summary: string, host_is_superhost: string, instant_bookable: string, host_total_listings_count: string, neighbourhood_cleansed: string, latitude: string, longitude: string, property_type: string, room_type: string, accommodates: string, bathrooms: string, bedrooms: string, beds: string, minimum_nights: string, number_of_reviews: string, review_scores_rating: string, review_scores_accuracy: string, review_scores_cleanliness: string, review_scores_checkin: string, review_scores_communication: string, review_scores_location: string, review_scores_value: string, price: string]

In [ ]:
display(fixed_price_df.summary().toPandas())

,summary,host_is_superhost,instant_bookable,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,...,minimum_nights,number_of_reviews,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,price
0,count,38039,38199,38194,38199,38199,38199,38199,38199,38199,...,38199,38199,26526,26494,26504,26490,26499,26487,26488,23256
1,mean,None,None,233.85301356233964,None,40.729101600620304,-73.94678742595275,None,None,2.795387313804026,...,29.164899604701695,24.95382078064871,4.731163763854399,4.771599230014364,4.661324328403318,4.835226500566287,4.832202724631156,4.745983312568484,4.655192162488713,185.2281131750946
2,stddev,None,None,930.5325521120192,None,0.056536312738795745,0.05450789978526201,None,None,1.9589800106021198,...,30.315807392295348,57.40776755810565,0.42064212726649597,0.4149460343227218,0.48702468795402876,0.3604325660625275,0.38756180932610934,0.37607022467551665,0.4587002432535944,152.3118917120247
3,min,f,f,1,Allerton,40.50031443485432,-74.251907,Barn,Entire home/apt,1,...,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
4,25%,None,None,1,None,40.68862,-73.9831232,None,None,2,...,30,0,4.65,4.71,4.53,4.81,4.82,4.65,4.55,82.0
5,50%,None,None,3,None,40.72639,-73.9543390342881,None,None,2,...,30,4,4.85,4.9,4.81,4.95,4.96,4.85,4.77,140.0
6,75%,None,None,12,None,40.76265,-73.92768083350654,None,None,4,...,30,22,5.0,5.0,5.0,5.0,5.0,5.0,4.94,239.0
7,max,t,t,8954,Woodside,40.91139,-73.71365,Tower,Shared room,16,...,1250,1909,5.0,5.0,5.0,5.0,5.0,5.0,5.0,999.0


# Getting rid of extreme values
Select only the `price` column and describe the result Dataframe in order to check the *min* and *max* values of the said column.

In [ ]:
display(fixed_price_df.select("price").describe().toPandas())

Now only keep rows with a strictly positive *price*.

In [ ]:
pos_prices_df = fixed_price_df.filter(<TODO>)

Let's take a look at the *min* and *max* values of the *minimum_nights* column:

In [ ]:
display(pos_prices_df.select("minimum_nights").describe().toPandas())

Group by minimum nights to check how many for each of them there are

In [ ]:
display(pos_prices_df
        .groupBy(<TODO>).<TODO>
        .orderBy(col("count").desc(), col("minimum_nights"))
       )

A minimum stay of one year seems to be a reasonable limit here. Let's filter out those records where the *minimum_nights* is greater than 365.

In [ ]:
min_nights_df = pos_prices_df.filter(<TODO>)

min_nights_df.show(truncate=False)

# Check null values for min nights

In [ ]:
print(f'total: {min_nights_df.count()}')
print(f'not nulls: {min_nights_df.na.drop().count()}')

#We have a lot with missing values, so not a good idea to remove them

## Handling Null Values
There are many different ways to handle null value, for instance you may just remove them, but sometines, null can actually be a key indicator of the thing you are trying to predict.
Some ways to handle nulls:
* Drop any records that contain nulls
* Numeric: Replace them with mean/median/zero/etc.
* Categorical: Replace them with the mode or create a special category for null (i.e. "absent")
* Use techniques like ALS (Alternating Least Squares) which are designed to impute missing values
  
**If you do ANY imputation techniques for categorical/numerical features, you MUST include an additional field specifying that field was imputed.**

## Spark Imputer 
 Spark imputer allows to impute values from null to another one, considering different strategies, it does not support imputation for categorical (string) features.
* <a href="https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.Imputer.html?highlight=imputer#pyspark.ml.feature.Imputer" target="_blank">Documentation.</a>
* <a href="https://spark.apache.org/docs/latest/ml-features.html#imputer" target="_blank">Programming guide.</a>

## Transformers and Estimators
Two key concepts introduced by the Spark ML API: **`transformers`** and **`estimators`**.
**Transformer**: Transforms one DataFrame into another DataFrame. It accepts a DataFrame as input, and returns a new DataFrame with one or more columns appended to it. Transformers do not learn any parameters from your data and simply apply rule-based transformations. It has a **`.transform()`** method.
**Estimator**: An algorithm which can be fit on a DataFrame to produce a Transformer. E.g., a learning algorithm is an Estimator which trains on a DataFrame and produces a model. It has a **`.fit()`** method because it learns (or "fits") parameters from your DataFrame.

In [ ]:
#Imputer sample
from pyspark.ml.feature import Imputer

df = spark.createDataFrame([
    (1.0, float("nan")),
    (2.0, float("nan")),
    (float("nan"), 3.0),
    (4.0, 4.0),
    (5.0, 5.0)
], ["a", "b"])


#Strategy mean (default), median, mode
imputer = Imputer(inputCols=["a", "b"], outputCols=["out_a", "out_b"])
model = imputer.fit(df)

model.transform(df).show()

In [ ]:
print(imputer.explainParams())

### Impute: Cast to Double
 SparkML's Imputer requires all fields to be of type double so we should cast all integer fields to double.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

integer_columns = [x.name for x in min_nights_df.schema.fields if x.dataType == IntegerType()]
doubles_df = min_nights_df

for c in integer_columns:
    doubles_df = doubles_df.withColumn(c, col(c).cast("double"))
    
columns = "\n - ".join(integer_columns)
print(f"Columns converted from Integer to Double:\n - {columns}")

Add an extra column to marks the presence of null values before imputing (i.e. 1.0 = Yes, 0.0 = No). This way we we'll have a way to know if a column was "fixed" or nor

In [ ]:
from pyspark.sql.functions import when

impute_cols = [
    'bedrooms',
    'bathrooms',
    'beds',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value'
]

for c in impute_cols:
    doubles_df = doubles_df.withColumn(<TODO>)

In [ ]:
display(doubles_df.describe().toPandas())

## Applying estimator
Now that all fields are double and we have an extra column for each of them indicating transformation we can apply the estimator
* Create an imputer with:
  * `"median"` strategy 
  * inputCols = impute_cols
  * outputCols same as inpute_cols so they are overriden
* Fit the doubles_df datarame to the imputer using the `fit` method
  * This will return a model
*  Use the previous model to transform the `doubles_df` to another `imputed_df` using the `transform` method

In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(<TODO>) 

imputer_model = imputer.fit(<TODO>) # compute strategy (median)
imputed_df = imputer_model.transform(<TODO>) # Apply calculation to the desired DF

## Saving output
Our data is cleansed now. Let's save this DataFrame to Delta using override

In [ ]:
imputed_df.write.mode('overwrite').parquet(output_path)